In [11]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
            # data_dft_d3bj = data_d3bj
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero

        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = (
                                data_scf[col[0]] - data_cc[col[0]]
                            )
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol")
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

print("Summary")
display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 12.609903921023943 kcal/mol, 116 in 1424849_W4_11
  -1 * W4_11-p4  -1 * -13.835936387768015  4 * W4_11-p  4 * -0.3065081166860182
Top 2 AI: 5.341987040214008 kcal/mol, 136 in 1424849_W4_11
  -1 * W4_11-foof  -1 * -4.328057378443191  2 * W4_11-f  2 * 0.6256503599724965  2 * W4_11-o  2 * -0.11868552908708807
Top 3 AI: 5.219463497924153 kcal/mol, 46 in 1424849_W4_11
  -1 * W4_11-alf3  -1 * -3.2680325650726445  1 * W4_11-al  1 * 0.07447985294857062  3 * W4_11-f  3 * 0.6256503599724965
Top 4 AI: 4.821493316645501 kcal/mol, 135 in 1424849_W4_11
  -1 * W4_11-cloo  -1 * -4.751233361486811  1 * W4_11-cl  1 * 0.30763101333286613  2 * W4_11-o  2 * -0.11868552908708807
Top 5 AI: 4.2865408250218024 kcal/mol, 63 in 1424849_W4_11
  -1 * W4_11-sif  -1 * -3.7919159706216305  1 * W4_11-si  1 * -0.13102550557232462  1 * W4_11-f  1 * 0.6256503599724965
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.61089226667

/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.

data_path       1424849                                          
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip
summary         0.32847      0.352175      0.031172       0.03176
W4_11          0.128193      0.146661      0.023594      0.021439
G21EA          0.079752      0.090296      0.012511      0.012762
G21IP          0.081996      0.086805      0.005995      0.006154
DIPCS10        0.101593      0.107438      0.005017      0.003622
PA26           0.268845        0.2982      0.034744      0.038429
SIE4x4         0.071736      0.093996      0.003569      0.001311
ALKBDE10       0.098409       0.10564      0.065934      0.061711
YBDE18         0.257196      0.276239      0.041119      0.041601
AL2X6          0.395256      0.396501      0.002764      0.002336
HEAVYSB11      0.288568      0.278481      0.011631      0.009043
NBPRC          0.237834      0.252724      0.032958      0.032482
ALK8           0.202733      0.187227      0.040614      0.040223
RC21           0.244294      0.258984      0.071539      0.082447
G2RC           0.163856      0.181463      0.015502      0.015617
BH76RC          0.12608       0.14036      0.055976      0.057338
FH51           0.342554      0.346443      0.024006      0.026834
TAUT15         0.369822      0.413544      0.067899      0.069453
DC13           0.390599      0.397856      0.020702      0.019322
MB16_43        0.500446      0.536587      0.084707      0.086338
DARC           0.426179      0.443789      0.014191      0.019082
RSE43          0.209548      0.233351      0.034582      0.033645
BSR36          0.534747      0.495778      0.004311      0.001683
CDIE20         0.370491       0.36033      0.039229      0.043548
ISO34           0.29061      0.298776      0.025594      0.026941
ISOL24              NaN           NaN           NaN           NaN
C60ISO              NaN           NaN           NaN           NaN
PArel          0.522926      0.599811      0.074776      0.079046
BH76            0.12608       0.14036      0.055976      0.057338
BHPERI         0.329315      0.326909      0.031284      0.029987
BHDIV10        0.342553      0.361694      0.049657      0.049067
INV24          0.778226      0.784024      0.038296      0.027518
BHROT27        0.289291      0.303021      0.021441      0.018507
PX13           0.215428      0.278637      0.012669       0.01315
WCPT18         0.197421      0.224154      0.060305      0.057861
RG18           0.204128      0.206046        0.0023      0.002641
ADIM6          0.480426      0.434301      0.000826       0.00024
S22            0.385426      0.405222      0.025222      0.024365
S66            0.331414      0.347655      0.022744      0.025355
HEAVY28        0.096282      0.106754      0.017323      0.009702
WATER27        0.393804      0.547345       0.02721      0.033217
CARBHB12       0.176663      0.195377      0.033396      0.036954
PNICO23        0.208847      0.233577        0.0258      0.023517
HAL59          0.399949      0.412527      0.042605      0.036969
AHB21          0.108228      0.129997      0.032458      0.031459
CHB6           0.162305       0.16005      0.019153      0.019259
IL16           0.290156      0.335812      0.049496       0.05021
IDISP           0.89541      0.816401      0.002265      0.000619
ICONF          0.464768      0.533359      0.023171      0.022986
ACONF          0.384884      0.355398      0.003603      0.001781
Amino20x4      0.723524      0.806256       0.02953      0.037327
PCONF21         1.05244      1.184712      0.058859      0.077448
MCONF          0.933391      0.987572      0.038508      0.045477
SCONF          0.606237      0.725383      0.033587      0.024507
UPU23               NaN           NaN           NaN           NaN
BUT14DIOL      0.338723      0.380054      0.025886       0.02199

MAE


data_path   1424849                                                      \
Disp type        AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       2.002477  13.710857  1.937471  13.77654  1.840461  13.713797   
sub2       5.510109   6.612164  5.571708  6.298111  6.192242   6.757732   
sub3       2.617318   6.261376  2.896715  6.594448  3.228945   6.934991   
sub4        1.93524   3.272197  1.534615  3.409243  2.900361   4.988172   
sub5       1.626727   1.321288  1.363798  0.843797  1.545785   0.872793   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

...

data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        3.239113   7.420922   2.890775     7.2373   2.651677   7.102094   
sub2       13.142366   12.53052  12.278113  11.460648  11.524607  10.005295   
sub3        4.832263   6.984991   5.141811   7.310673   5.522858   7.521852   
sub4       11.556831  11.029773   7.213895   7.031697  12.876599  13.798614   
sub5       15.335367  11.571826  12.618257   7.156504  14.544885   7.523212   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

wtmad_2


data_path   1424849                                                     \
Disp type        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       1.438234  4.040696  1.327334  4.010375  1.263846   3.989769   
sub2       3.339073  3.635331  2.947971  3.094352  2.735086   2.545688   
sub3       1.303233  2.611064  1.410302   2.74261  1.549133   2.873647   
sub4       4.683102  4.159493  3.487294  3.039102  6.578153    6.62018   
sub5       6.331811   4.66236  5.508358  3.059454  6.387196    3.19725   

data_path            
Disp type Processed  
sub1        18 / 18  
sub2          7 / 9  
sub3          7 / 7  
sub4        11 / 12  
sub5          8 / 9

Summary of Subset
MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       1.028249  29.506863   1.329678  29.939142   1.248828  29.861054   
G21EA       0.816732   9.754522   0.816446   9.754712   0.823677   9.757546   
G21IP       0.694714   8.953334   0.696809   8.951788   0.685192     8.9519   
DIPCS10     2.042735  12.310852   2.039187  12.314833   2.049052  12.264655   
PA26        3.009087   2.200766   2.979381   2.064804   2.951891    2.00824   
SIE4x4      3.430805  21.908516   3.569856  22.087662   3.729847  22.339928   
ALKBDE10    0.959132  18.125246   1.012348  18.251661    0.94979  18.135588   
YBDE18      5.939045   8.145393   5.005577   7.864281   4.255001   7.472761   
AL2X6       6.140013   5.659145   3.948525   3.700047    2.63339   1.829895   
HEAVYSB11   2.024005   5.391476   1.847266   5.349873   1.845095   5.984154   
NBPRC       3.058861    2.23255   2.588226   1.879514   2.192134   2.274234   
ALK8        4.403777   4.400063   2.883759   3.468354   2.009981   2.559066   
RC21        2.456062   4.820926   1.796951   5.349599   1.816792     6.0297   
G2RC        1.887093   5.917203   2.020313   6.219125   2.145182   6.417298   
BH76RC       0.85212   3.484561   0.904135   3.498541    0.96494   3.536982   
FH51        2.464205   3.703657   2.122457   3.450819   2.136612   3.256908   
TAUT15      1.702465    2.16453   1.652523   2.155472   1.578771   2.145937   
DC13        6.170057  13.090861   6.288349  12.515441   5.746533  11.475687   
MB16_43     13.39941  15.461604  16.570448  17.561642  21.115024  23.165472   
DARC        7.823359   10.75415   4.976004   8.032632   3.060175    5.57797   
RSE43        3.11738     3.1576   3.020871   3.057761   2.755026    2.74911   
BSR36        4.98276    8.40118   2.934988   5.428274   2.368872   3.079317   
CDIE20      1.450234   1.599507     1.4536   1.517479   1.479584   1.347484   
ISO34       2.020035   2.001743   1.874011   1.851077   1.708557   1.647042   
ISOL24             0          0          0          0          0          0   
C60ISO             0          0          0          0          0          0   
PArel       3.015433   1.743933   2.976004   1.740399   2.907704   1.645025   
BH76        2.803583   9.107946   2.898571   9.363941   3.093517   9.676053   
BHPERI      2.657111   3.268992   4.013655   4.768458   5.491745     6.4139   
BHDIV10     4.741552   6.277706   4.881882   6.541159    4.85035   6.535109   
INV24       3.393792   2.697333   3.555006   2.443017   3.631489   2.228547   
BHROT27     1.713103   0.853379   1.719478   0.843699   1.763036    0.78352   
PX13        1.059156  11.042023   1.065327  11.264187   1.212867  11.219857   
WCPT18      2.039614   7.967151   2.383461   8.356955   2.749679   8.744304   
RG18         0.30936   0.230018   0.386527   0.241058   0.819514   0.709552   
ADIM6       3.598266   3.059029   0.809334   0.168486   2.777618   2.390824   
S22         2.728845   2.291252   2.098021   1.087525   3.196728   2.291758   
S66         2.326488   1.894709   0.793497   0.760268   1.834235   2.101976   
HEAVY28            0          0          0          0          0          0   
WATER27      3.02651  16.299413   3.597118  20.504456   9.177355  26.402884   
CARBHB12    0.484944   1.632408   0.535487   2.191305   1.030009   2.841777   
PNICO23     0.840943   0.776429   1.062236   1.170506    1.44661   1.726817   
HAL59       2.035403   1.649786   1.835204   1.408027   2.562614   2.247276   
AHB21       1.666768   2.453821   1.898014   2.804963   2.212706   3.294519   
CHB6        1.766454    1.80532   1.664622   1.984716   1.432927   2.309161   
IL16        1.467889   1.021067   2.274265   2.368045   4.059287   4.409963   
IDISP      13.356085  13.125475    9.12502   7.264733   6.653317   5.245746   
ICONF       0.946016   0.413804   0.911044    0.40406   0.905493   0.493337   
ACONF       3.016487   0.546621   2.711755   0.158

In [5]:
4.04055 + 3.63395 + 2.611338 + 4.154765 + 4.658355


19.098958

In [6]:
1.379994 + 2.970755 + 1.66299 + 6.811273 + 6.30221

19.127222

In [10]:
array1 = np.array(
    [
        [-0.11669382215923368, -0.04046288967220035, 1.0783136152669788],
        [-0.7186060510091714, 0.699161310349129, -0.415684678578355],
        [-0.17048761174944466, -0.9863427442578072, -0.4196151945900501],
        [1.0057874849178496, 0.3276443235808786, -0.24301374209857368],
    ]
)

for i in range(array1.shape[0]):
    print(np.sum(array1[i, :] ** 2))
    for j in range(i):
        # print angle between i and j
        dot_product = np.dot(array1[i, :], array1[j, :])
        norm_i = np.linalg.norm(array1[i, :])
        norm_j = np.linalg.norm(array1[j, :])
        angle = np.arccos(dot_product / (norm_i * norm_j))
        print(f"Angle between {i} and {j}: {np.degrees(angle):.2f} degrees")



1.1780149464408975
1.1780149464408973
Angle between 1 and 0: 109.47 degrees
1.178014946440897
Angle between 2 and 0: 109.47 degrees
Angle between 2 and 1: 109.47 degrees
1.178014946440897
Angle between 3 and 0: 109.47 degrees
Angle between 3 and 1: 109.47 degrees
Angle between 3 and 2: 109.47 degrees
